In [32]:
import pandas as pd

train_files = [
    "data/CRMLSSold202506.csv",
    "data/CRMLSSold202507.csv",
    "data/CRMLSSold202508.csv",
    "data/CRMLSSold202509.csv",
    "data/CRMLSSold202510.csv",
    "data/CRMLSSold202511.csv",
    "data/CRMLSSold202512.csv",
    "data/CRMLSSold202601.csv",
    "data/CRMLSSold202602.csv",
    "data/CRMLSSold202603.csv",
    "data/CRMLSSold202604.csv"
]
test_file = "data/CRMLSSold202605.csv"

# Load training set
df_train = pd.concat([pd.read_csv(f, low_memory=False) for f in train_files], ignore_index=True)

df_train = df_train[
    (df_train["PropertyType"] == "Residential") &
    (df_train["PropertySubType"] == "SingleFamilyResidence")
]
print(df_train.shape)
#print(df_train.info())

# Load test set
df_test = pd.read_csv(test_file)

df_test = df_test[
    (df_test["PropertyType"] == "Residential") &
    (df_test["PropertySubType"] == "SingleFamilyResidence")
]
print(df_test.shape)
#print(df_test.info())

(118196, 78)
(12024, 78)


Handle Missing Values

In [33]:
key_cols = ["ClosePrice", "LivingArea", "BedroomsTotal", "BathroomsTotalInteger", "LotSizeSquareFeet"]

summary = pd.DataFrame({
    "Data Type": df_train[key_cols].dtypes,
    "Missing Count": df_train[key_cols].isnull().sum(),
    "Missing %": (df_train[key_cols].isnull().mean() * 100).round(2)
})

summary_test = pd.DataFrame({
    "Data Type": df_test[key_cols].dtypes,
    "Missing Count": df_test[key_cols].isnull().sum(),
    "Missing %": (df_test[key_cols].isnull().mean() * 100).round(2)
})

print("Summary for training:", summary)
print("\nSummary for testing:", summary_test)

Summary for training:                       Data Type  Missing Count  Missing %
ClosePrice              float64              0       0.00
LivingArea              float64             61       0.05
BedroomsTotal           float64              0       0.00
BathroomsTotalInteger   float64             12       0.01
LotSizeSquareFeet       float64           2032       1.72

Summary for testing:                       Data Type  Missing Count  Missing %
ClosePrice              float64              0       0.00
LivingArea              float64              7       0.06
BedroomsTotal           float64              0       0.00
BathroomsTotalInteger   float64              0       0.00
LotSizeSquareFeet       float64            204       1.70


- Impute the missing values with the median computed from the training set for LivingArea, BathroomsTotalInteger, and LotSizeSquareFeet, since missing rate for each is less than 5%.
- Also flag missing for LotSizeSquareFeet.

In [34]:
# For Training Data
cleaned_train = df_train.copy()

# Flag missing for LotSizeSquareFeet
cleaned_train["Missing_LotSizeSquareFeet"] = cleaned_train["LotSizeSquareFeet"].isnull().astype(int)

# Impute missing values
for col in ["LivingArea", "BathroomsTotalInteger", "LotSizeSquareFeet"]:
    median_val = cleaned_train[col].median()
    cleaned_train[col] = cleaned_train[col].fillna(median_val)


# For Test Data
cleaned_test = df_test.copy()

# Flag missing for LotSizeSquareFeet
cleaned_test["Missing_LotSizeSquareFeet"] = cleaned_test["LotSizeSquareFeet"].isnull().astype(int)

# Impute missing values
for col in ["LivingArea", "LotSizeSquareFeet"]:
    median_val = cleaned_train[col].median() # Use median from training data
    cleaned_test[col] = cleaned_test[col].fillna(median_val)


Convert Categorical Fields

In [35]:
non_numeric_cols = cleaned_train.select_dtypes(exclude="number").columns
print(cleaned_train[non_numeric_cols].dtypes)

BuyerAgentAOR                  str
ListAgentAOR                   str
Flooring                       str
ViewYN                      object
WaterfrontYN                object
BasementYN                  object
PoolPrivateYN               object
ListAgentEmail                 str
CloseDate                      str
ListAgentFirstName             str
ListAgentLastName              str
UnparsedAddress                str
PropertyType                   str
ListOfficeName                 str
BuyerOfficeName                str
CoListOfficeName               str
ListAgentFullName              str
CoListAgentFirstName           str
CoListAgentLastName            str
BuyerAgentMlsId                str
BuyerAgentFirstName            str
BuyerAgentLastName             str
AssociationFeeFrequency        str
MLSAreaMajor                   str
CountyOrParish                 str
MlsStatus                      str
ElementarySchool               str
AttachedGarageYN            object
BuilderName         

1. Convert CloseDate, ContractStatusChangeDate , PurchaseContractDate, ListingContractDate to datetime 

In [36]:
date_cols = ["CloseDate", "ContractStatusChangeDate" , "PurchaseContractDate", "ListingContractDate"]

for col in date_cols:
    cleaned_train[col] = pd.to_datetime(cleaned_train[col], errors="coerce")
    cleaned_test[col] = pd.to_datetime(cleaned_test[col], errors="coerce")

cleaned_train[date_cols].info()
cleaned_test[date_cols].info()

<class 'pandas.DataFrame'>
Index: 118196 entries, 3 to 235382
Data columns (total 4 columns):
 #   Column                    Non-Null Count   Dtype         
---  ------                    --------------   -----         
 0   CloseDate                 118196 non-null  datetime64[us]
 1   ContractStatusChangeDate  118196 non-null  datetime64[us]
 2   PurchaseContractDate      118190 non-null  datetime64[us]
 3   ListingContractDate       118196 non-null  datetime64[us]
dtypes: datetime64[us](4)
memory usage: 4.5 MB
<class 'pandas.DataFrame'>
Index: 12024 entries, 1 to 23256
Data columns (total 4 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   CloseDate                 12024 non-null  datetime64[us]
 1   ContractStatusChangeDate  12024 non-null  datetime64[us]
 2   PurchaseContractDate      12023 non-null  datetime64[us]
 3   ListingContractDate       12024 non-null  datetime64[us]
dtypes: datetime6

In [37]:
# Extract Date Features 
cleaned_train["CloseMonth"] = cleaned_train["CloseDate"].dt.month
cleaned_train["CloseYear"] = cleaned_train["CloseDate"].dt.year
cleaned_train["CloseDay"] = cleaned_train["CloseDate"].dt.date

cleaned_test["CloseMonth"] = cleaned_test["CloseDate"].dt.month
cleaned_test["CloseYear"] = cleaned_test["CloseDate"].dt.year
cleaned_test["CloseDay"] = cleaned_test["CloseDate"].dt.date

2. Perform One-Hot Encoding on Selected Categotical Columns

In [38]:
categorical_cols = ["City", "PostalCode", "StateOrProvince", "CountyOrParish", "HighSchoolDistrict"]
print(cleaned_train[categorical_cols].head())

print("\nNumber of unique values")
for col in categorical_cols:
    print(f"{col}: ", cleaned_train[col].nunique())

              City PostalCode StateOrProvince  CountyOrParish  \
3   Lake Arrowhead      92352              CA  San Bernardino   
10     Los Angeles      90046              CA     Los Angeles   
11    Hillsborough      94010              CA       San Mateo   
13     Yorba Linda      92886              CA          Orange   
14   Castro Valley      94546              CA         Alameda   

               HighSchoolDistrict  
3                Rim of the World  
10                            NaN  
11                          Other  
13  Placentia-Yorba Linda Unified  
14                            NaN  

Number of unique values
City:  978
PostalCode:  2016
StateOrProvince:  4
CountyOrParish:  60
HighSchoolDistrict:  421


- Only one-hot encode on CountyOrParish and top 200 City and PostalCode

In [39]:
top_cities = cleaned_train["City"].value_counts().nlargest(200).index
top_postal = cleaned_train["PostalCode"].value_counts().nlargest(200).index

# Replace cities and postal codes not in top 200 with "other"
# On training set
cleaned_train["City_grouped"] = cleaned_train["City"].apply(
    lambda x: x if x in top_cities else "Other"
)
cleaned_train["PostalCode_grouped"] = cleaned_train["PostalCode"].apply(
    lambda x: x if x in top_postal else "Other"
)

# On test set
cleaned_test["City_grouped"] = cleaned_test["City"].apply(
    lambda x: x if x in top_cities else "Other"
)
cleaned_test["PostalCode_grouped"] = cleaned_test["PostalCode"].apply(
    lambda x: x if x in top_postal else "Other"
)

categorical_cols = ["City_grouped", "PostalCode_grouped", "CountyOrParish"]
cleaned_train = cleaned_train.drop(columns=["City", "PostalCode"])
cleaned_test = cleaned_test.drop(columns=["City", "PostalCode"])

In [40]:
cleaned_train.head()

,BuyerAgentAOR,ListAgentAOR,Flooring,ViewYN,WaterfrontYN,BasementYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,...,HighSchoolDistrict,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,Missing_LotSizeSquareFeet,CloseMonth,CloseYear,CloseDay,City_grouped,PostalCode_grouped
3,TheInlandGateway,TheInlandGateway,NaN,True,NaN,NaN,False,889000.0,523319952,hutton@cbappteam.com,...,Rim of the World,0.0,9600.0,NaN,0,6,2025,2025-06-13,Lake Arrowhead,92352
10,BeverlyHillsGreaterLa,BeverlyHillsGreaterLa,Wood,True,NaN,NaN,False,1899999.0,1118606385,chase.campen@compass.com,...,NaN,NaN,10400.0,NaN,0,6,2025,2025-06-30,Los Angeles,90046
11,Mlslistings,Mlslistings,NaN,False,NaN,NaN,NaN,NaN,1118606192,stanleylo@greenbanker.com,...,Other,NaN,22505.0,NaN,0,6,2025,2025-06-30,Other,Other
13,PacificWest,PacificWest,NaN,True,NaN,NaN,False,865000.0,1118604114,matt@majorleaguesocal.com,...,Placentia-Yorba Linda Unified,0.0,4800.0,NaN,0,6,2025,2025-06-30,Yorba Linda,92886
14,BayEast,BayEast,"Carpet,Laminate",NaN,NaN,NaN,False,875000.0,1118603794,brianrowland.homes@gmail.com,...,NaN,NaN,5500.0,NaN,0,6,2025,2025-06-30,Castro Valley,94546


In [ ]:
# One-hot encoding
from sklearn.preprocessing import OneHotEncoder

onehotencoder = OneHotEncoder(handle_unknown="ignore", sparse_output=True)

train_encoded = onehotencoder.fit_transform(cleaned_train[categorical_cols])
test_encoded = onehotencoder.transform(cleaned_test[categorical_cols])

In [56]:
from scipy import sparse

X_train = sparse.hstack([cleaned_train.drop(columns=categorical_cols).select_dtypes(include="number").values, train_encoded])
X_test = sparse.hstack([cleaned_test.drop(columns=categorical_cols).select_dtypes(include="number").values, test_encoded])

# Convert into dataframe
feature_names = list(cleaned_train.drop(columns=categorical_cols).select_dtypes(include="number").columns) + list(onehotencoder.get_feature_names_out(categorical_cols))

X_train_df = pd.DataFrame.sparse.from_spmatrix(X_train, columns=feature_names)
X_test_df = pd.DataFrame.sparse.from_spmatrix(X_test, columns=feature_names)

print("\nTraining set shape: ", X_train_df.shape)
print("Test set shape: ", X_test_df.shape)



Training set shape:  (118196, 495)
Test set shape:  (12024, 495)


Save Cleaned CSV

In [ ]:
X_train_df.to_csv("data_cleaned/cleaned_train.csv", index=False)
X_test_df.to_csv("data_cleaned/cleaned_test.csv", index=False)